In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

In [ ]:
class MeetingState(TypedDict):
    user_query: str
    attendees: List[str]      # multiple entities extracted at once
    date: str
    duration_minutes: int
    availability: dict        # {name: True/False}
    all_available: bool
    proposed_slot: str
    message: str

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

model = init_chat_model(
    "openai:gpt-5.4-nano",
    temperature=0
)

In [ ]:
import json

def understand_request(state: MeetingState):
    print("Extracting meeting details...")

    prompt = f"""
    Extract meeting scheduling details from this request:

    {state["user_query"]}

    Return JSON with these fields:
    - attendees: a list of attendee names (array of strings)
    - date: the requested date
    - duration_minutes: meeting length in minutes (integer)
    """

    response = model.invoke(prompt)
    text = response.content.replace("```json", "").replace("```", "").strip()
    data = json.loads(text)

    return {
        "attendees": data.get("attendees", []),
        "date": data.get("date"),
        "duration_minutes": data.get("duration_minutes")
    }

In [ ]:
def check_availability(state: MeetingState):
    print("Checking availability for each attendee...")

    # Dummy check — in a real system this would hit a calendar API per person
    availability = {name: True for name in state["attendees"]}

    # simulate one person being busy, just to exercise the branching logic
    if len(state["attendees"]) > 2:
        availability[state["attendees"][-1]] = False

    all_available = all(availability.values())

    return {
        "availability": availability,
        "all_available": all_available
    }

In [ ]:
def propose_slot(state: MeetingState):
    print("Proposing meeting slot...")

    return {
        "proposed_slot": f"{state['date']} for {state['duration_minutes']} minutes"
    }

In [ ]:
def suggest_alternative(state: MeetingState):
    print("Someone is unavailable, suggesting alternative...")

    busy_people = [name for name, free in state["availability"].items() if not free]

    return {
        "message": f"Cannot schedule — {', '.join(busy_people)} unavailable on {state['date']}. Suggest a different date."
    }

In [ ]:
def confirm_booking(state: MeetingState):
    print("Confirming meeting...")

    return {
        "message": f"Meeting confirmed for {', '.join(state['attendees'])} on {state['proposed_slot']}."
    }

In [ ]:
graph = StateGraph(MeetingState)

graph.add_node("understand_request", understand_request)
graph.add_node("check_availability", check_availability)
graph.add_node("propose_slot", propose_slot)
graph.add_node("suggest_alternative", suggest_alternative)
graph.add_node("confirm_booking", confirm_booking)

In [ ]:
graph.add_edge(START, "understand_request")
graph.add_edge("understand_request", "check_availability")

# conditional branch: route based on all_available
def route_after_check(state: MeetingState):
    return "propose_slot" if state["all_available"] else "suggest_alternative"

graph.add_conditional_edges(
    "check_availability",
    route_after_check,
    {
        "propose_slot": "propose_slot",
        "suggest_alternative": "suggest_alternative"
    }
)

graph.add_edge("propose_slot", "confirm_booking")
graph.add_edge("confirm_booking", END)
graph.add_edge("suggest_alternative", END)  # dead-end if someone's busy

In [ ]:
app = graph.compile()

In [ ]:
from IPython.display import Image, display

display(
    Image(
        app.get_graph().draw_mermaid_png()
    )
)

In [ ]:
result = app.invoke({
    "user_query": "Schedule a meeting with Alice and Bob on Friday for 30 minutes"
})
print(result)

In [ ]:
result = app.invoke({
    "user_query": "Schedule a meeting with Alice, Bob, and Charlie on Monday for 45 minutes"
})
print(result)

In [ ]:
from pprint import pprint

result = app.invoke({
    "user_query": "Schedule a meeting with Alice, Bob, and Charlie on Monday for 45 minutes"
})

pprint(result)

In [ ]:
from pprint import pprint

print("Final State:")
pprint(result, width=80, sort_dicts=False)